# 01. Bag of Words - reproducible training

**Task:** Topic Classification (20-Newsgroups, 4 classes)
**Framework:** scikit-learn
**Real implementation:** CountVectorizer (5k features, 1-2 grams, English stop-words) + Logistic Regression

## GPU?
**Not required.** This is a small classical NLP model that trains on CPU in
seconds to ~1 minute. A free Colab CPU runtime is enough; a GPU will not make it
meaningfully faster.

Run the cells in order. At the end you will have a fresh
`01_bag_of_words/artifacts/model.joblib` plus verified metrics and an unseen-text test.


## 1. Get the repository


In [ ]:
# Clone the portfolio (or upload your own copy and skip this cell).
import os
REPO_DIR = "nlp-models"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/deepvisionkararhaider-crypto/nlp-models.git
%cd {REPO_DIR}
!git rev-parse --short HEAD


## 2. Install dependencies


In [ ]:
!pip install -q -r 01_bag_of_words/requirements.txt
import sklearn, numpy, pandas
print("scikit-learn", sklearn.__version__, "| numpy", numpy.__version__)


## 3. Data

The four-category dataset (`data/20newsgroups_4cat/`) is committed to the
repository. If it is missing, rebuild it from the original 20-Newsgroups archive:

```bash
python training/build_dataset.py --archive /path/to/20news-bydate.tar.gz
```


In [ ]:
import os
print("dataset present:", os.path.exists("data/20newsgroups_4cat/train.csv"))
import pandas as pd
if os.path.exists("data/20newsgroups_4cat/train.csv"):
    df = pd.read_csv("data/20newsgroups_4cat/train.csv")
    print(df.shape, "| classes:", df["category"].unique().tolist())


## 4. Train (and save artifacts)


In [ ]:
# Trains this model's original architecture and writes artifacts/ + metrics.json
!python training/train_all.py --only 1


## 5. Evaluate on the held-out test set


In [ ]:
import json
met = json.load(open("01_bag_of_words/artifacts/metrics.json"))
print("Test accuracy :", round(met["accuracy"], 4))
print("F1 (macro)    :", round(met["f1_macro"], 4))
print("N test        :", met["n_test"])
print()
for cls, v in met["per_class"].items():
    print(f"  {cls:<10} precision={v['precision']:.3f} recall={v['recall']:.3f} f1={v['f1']:.3f}")
print()
print("Confusion matrix (rows=true, cols=pred):")
for row in met["confusion_matrix"]:
    print(" ", row)


## 6. Inference test on **new, unseen** text


In [ ]:
import sys; sys.path.insert(0, ".")
from common import nlp_common as nc

UNSEEN = [
    ("hockey",   "The goalie made an incredible save in the third period and the team reached the playoffs."),
    ("space",    "NASA launched a new rocket to the space station to study the atmosphere of Mars."),
    ("graphics", "The 3D rendering engine uses ray tracing for realistic shadows and textures."),
    ("politics", "The senator debated the tax policy and voting reform bill before the election."),
]
correct = 0
for expected, text in UNSEEN:
    r = nc.predict_text("01_bag_of_words", text)
    ok = r["predicted_short"] == expected
    correct += ok
    print(f"{'OK ' if ok else '== '} expected={expected:<9} got={r['predicted_short']:<9} "
          f"score={r['scores'][0]['score']:.3f} ({r['score_type']})")
print(f"\nunseen accuracy: {correct}/{len(UNSEEN)}")


## 7. Export the artifacts


In [ ]:
import shutil
shutil.make_archive("artifacts_01_bag_of_words", "zip", f"01_bag_of_words/artifacts")
print("wrote artifacts_01_bag_of_words.zip - download it from the Files panel.")

# Optional: commit straight back to your fork
# !git add 01_bag_of_words/artifacts && git commit -m "Retrain Bag of Words" && git push


## 8. Where the artifacts go

Place the produced `model.joblib` (and `metrics.json`, `meta.json`) in
**`01_bag_of_words/artifacts/`** in the GitHub repository, then deploy
`01_bag_of_words/app.py` on Streamlit Community Cloud.

> Some models (Word2Vec / FastText) also write a large gensim model file
> (`w2v.model` / `ft.model`) inside `artifacts/`. Keep it next to `model.joblib`
> - the app loads both.
